# Direct Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local Direct baseline for English-to-Chinese cross-lingual dialogue summarization.

The Direct pipeline uses a single local small language model agent. The agent reads the original English dialogue and directly generates a concise Chinese summary without using an intermediate English summary, full-dialogue translation, semantic representation, or revision step.

```text
English Dialogue
→ Direct Summarization Agent
→ Final Chinese Summary
```

The pipeline consists of one agent:

```text
Direct Summarization Agent
Input: original English dialogue
Output: concise Chinese summary
```
This setup is used as the simplest cross-lingual summarization baseline. Unlike Translate-then-Summarize, Summarize-then-Translate, or the Semantic-agent pipeline, the Direct pipeline performs the task in a single model call.

The local small language model is served through Ollama. The notebook controls the prompt design, input/output processing, direct summary generation, output inspection, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the local model used for the Direct baseline.

### Recommended model setup

This notebook uses one local small language model through Ollama.

```bash
ollama pull qwen3.5:27b
```

If qwen3.5:27b is too slow on your machine, you can use a smaller model for testing:

```bash
ollama pull "qwen3.5:9b"
```

```bash
DIRECT_MODEL = "qwen3.5:27b"
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [3]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [8]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# Direct model
DIRECT_MODEL = "qwen3.5:27b"  # Change this if your local Ollama model name is different

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# Gold set path
GOLD_SET_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_set_50_zh_XSAMSum_bart.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files for Direct baseline
FULL_OUTPUT_PATH = OUTPUT_DIR / "direct_qwen27b_5samples.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "direct_qwen27b_5samples.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "direct_qwen27b_5samples_errors.jsonl"

print("Gold set path:", GOLD_SET_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Gold set path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_set_50_zh_XSAMSum_bart.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_5samples.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_5samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_5samples_errors.jsonl


In [9]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['qwen3.5:9b', 'qwen3.5:27b']


In [10]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [11]:
# Cell 5: Direct prompt template

DIRECT_PROMPT = """You are a cross-lingual dialogue summarization agent.

Your task is to read the following English dialogue and directly generate a concise Chinese summary.

Requirements:
- Generate the summary directly in Chinese.
- Do not first write an English summary.
- Do not translate the full dialogue sentence by sentence.
- Summarize only the main information and final outcome.
- Keep the summary concise and faithful to the dialogue.
- Do not add information that is not stated or clearly implied.
- Do not explain your reasoning.
- Output only the final Chinese summary.

English dialogue:
{dialogue}

Chinese summary:
"""

In [12]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [13]:
# Cell 7: Direct agent function

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace named placeholders in the prompt."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def direct_agent(dialogue: str) -> str:
    """Direct baseline: English dialogue -> Chinese summary."""
    prompt = fill_prompt(
        DIRECT_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=DIRECT_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [14]:
# Cell 8: Direct pipeline

def run_direct_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run the Direct pipeline: English dialogue -> Chinese summary."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    final_chinese_summary = direct_agent(dialogue)

    return {
        "id": sample_id,
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,
        "final_summary": final_chinese_summary,
        "pipeline": "direct",
        "model": DIRECT_MODEL,
        "num_model_calls": 1,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [15]:
# Cell 9: Load first 5 examples from the gold set

def load_examples_from_gold_set(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the first n examples from the gold-set JSON file."""
    if not path.exists():
        raise FileNotFoundError(f"Gold set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"gold_{i+1:05d}",
            "test_index": item.get("test_index", ""),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_gold_set(GOLD_SET_PATH, n=5)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'gold_00001', 'test_index': 59, 'dialogue': "Laura: Where are you?\r\nPaul: Almost there.\r\nLaura: Which is?\r\nPaul: Close to the Mac.\r\nLaura: That's so far away!\r\nPaul: 15 mins\r\nLaura: I am not waiting any more, see you some other time.\r\nPaul: Please, wait!\r\nLaura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.\r\nPaul: I am so sorry.\r\nLaura: I am not. ", 'reference_english_summary': 'Paul is late for a meeting with Laura and she refuses to wait any longer.', 'reference_chinese_summary': '保罗和劳拉见面时迟到了，现在劳拉不想再等了。'}


In [16]:
# Cell 10: Run the Direct pipeline for the first example

result = run_direct_pipeline(test_data[0])
result

{'id': 'gold_00001',
 'test_index': 59,
 'dialogue': "Laura: Where are you?\r\nPaul: Almost there.\r\nLaura: Which is?\r\nPaul: Close to the Mac.\r\nLaura: That's so far away!\r\nPaul: 15 mins\r\nLaura: I am not waiting any more, see you some other time.\r\nPaul: Please, wait!\r\nLaura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.\r\nPaul: I am so sorry.\r\nLaura: I am not. ",
 'reference_english_summary': 'Paul is late for a meeting with Laura and she refuses to wait any longer.',
 'reference_chinese_summary': '保罗和劳拉见面时迟到了，现在劳拉不想再等了。',
 'final_summary': '保罗因迟到且沟通位置不清，导致劳拉在等待 30 分钟后决定取消见面，尽管保罗道歉，劳拉仍坚持离开。',
 'pipeline': 'direct',
 'model': 'qwen3.5:27b',
 'num_model_calls': 1}

In [17]:
# Cell 11: Print Direct pipeline result clearly

def print_direct_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Direct Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Model:", result["model"])
    print("Model calls:", result["num_model_calls"])


print_direct_result(result)

=== Original Dialogue ===
Laura: Where are you?
Paul: Almost there.
Laura: Which is?
Paul: Close to the Mac.
Laura: That's so far away!
Paul: 15 mins
Laura: I am not waiting any more, see you some other time.
Paul: Please, wait!
Laura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.
Paul: I am so sorry.
Laura: I am not. 

=== Direct Chinese Summary ===
保罗因迟到且沟通位置不清，导致劳拉在等待 30 分钟后决定取消见面，尽管保罗道歉，劳拉仍坚持离开。

=== Reference English Summary ===
Paul is late for a meeting with Laura and she refuses to wait any longer.

=== Reference Chinese Summary ===
保罗和劳拉见面时迟到了，现在劳拉不想再等了。

=== Metadata ===
Pipeline: direct
Model: qwen3.5:27b
Model calls: 1


## 4. Save Results

This saves all intermediate outputs and the final output.


In [18]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_5samples.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_5samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_5samples_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [ ]:
# Cell 13: Batch inference with Direct pipeline
# Time stamp: 46.3s

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running Direct pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_direct_pipeline(ex)
        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "test_index": ex.get("test_index", ""),
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running Direct pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Finished. Outputs saved to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_5samples.jsonl


## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [20]:
# Cell 14: Export Direct summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "test_index": record.get("test_index", ""),
        "dialogue": record.get("dialogue", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),
        "pipeline": record.get("pipeline", "direct"),
        "model": record.get("model", ""),
        "num_model_calls": record.get("num_model_calls", 1),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_5samples.csv


,id,test_index,dialogue,final_summary,reference_english_summary,reference_chinese_summary,pipeline,model,num_model_calls
0,gold_00001,59,Laura: Where are you?\r\nPaul: Almost there.\r...,保罗因迟到且沟通不清，导致劳拉在等待 30 分钟后决定取消见面，尽管保罗道歉，劳拉仍拒绝原谅。,Paul is late for a meeting with Laura and she ...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。,direct,qwen3.5:27b,1
1,gold_00002,81,Finn: Hey\r\nZadie: Hi there! What's up?\r\nFi...,Finn 邀请 Zadie 明天下午 2 点前往 Elephant and Castle 购...,Finn and Zadie are going to Elephant and Castl...,费恩和查蒂明天2点去象堡，他们会在正门碰头。,direct,qwen3.5:27b,1
2,gold_00003,85,Josh: I need to buy an iPad?\r\nJosh: do u thi...,Josh 咨询购买 iPad 的建议，Brian 不推荐苹果品牌，转而推荐三星、小米或索尼，...,Josh wants to buy a tablet and doesn't know wh...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...,direct,qwen3.5:27b,1
3,gold_00004,87,Frank: wat are u doing??\r\nAndy: watching Arr...,Frank 提醒 Andy 明天有测验，劝他立即复习，但 Andy 认为测验不重要并计划明天...,Frank tries to encourage Andy to learn for the...,弗兰克试图激励安迪，为明天的测验学习。,direct,qwen3.5:27b,1
4,gold_00005,123,Crystal: <file_photo>\r\nIrene: He's so big!\r...,Crystal 分享孩子长大的照片，感叹孩子因长高而衣服不合身。Irene 主动提出带孩子去...,Irene will take Crystal's son shopping for clo...,艾琳会带克里斯特尔的儿子去买衣服。,direct,qwen3.5:27b,1


In [21]:
# Cell 15: Compare generated Chinese summary with the reference Chinese summary

comparison_columns = [
    "id",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,final_summary,reference_chinese_summary
0,gold_00001,保罗因迟到且沟通不清，导致劳拉在等待 30 分钟后决定取消见面，尽管保罗道歉，劳拉仍拒绝原谅。,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,Finn 邀请 Zadie 明天下午 2 点前往 Elephant and Castle 购...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,Josh 咨询购买 iPad 的建议，Brian 不推荐苹果品牌，转而推荐三星、小米或索尼，...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,Frank 提醒 Andy 明天有测验，劝他立即复习，但 Andy 认为测验不重要并计划明天...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,Crystal 分享孩子长大的照片，感叹孩子因长高而衣服不合身。Irene 主动提出带孩子去...,艾琳会带克里斯特尔的儿子去买衣服。


In [22]:
# Cell 15: Compare generated Chinese summary with the reference Chinese summary

comparison_columns = [
    "id",
    "test_index",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,test_index,final_summary,reference_chinese_summary
0,gold_00001,59,保罗因迟到且沟通不清，导致劳拉在等待 30 分钟后决定取消见面，尽管保罗道歉，劳拉仍拒绝原谅。,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,81,Finn 邀请 Zadie 明天下午 2 点前往 Elephant and Castle 购...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,85,Josh 咨询购买 iPad 的建议，Brian 不推荐苹果品牌，转而推荐三星、小米或索尼，...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,87,Frank 提醒 Andy 明天有测验，劝他立即复习，但 Andy 认为测验不重要并计划明天...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,123,Crystal 分享孩子长大的照片，感叹孩子因长高而衣服不合身。Irene 主动提出带孩子去...,艾琳会带克里斯特尔的儿子去买衣服。


In [23]:
# Cell 16: Inspect Direct outputs

if not df.empty:
    inspection_columns = [
        "id",
        "test_index",
        "dialogue",
        "final_summary",
        "reference_chinese_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,test_index,dialogue,final_summary,reference_chinese_summary
0,gold_00001,59,Laura: Where are you?\r\nPaul: Almost there.\r...,保罗因迟到且沟通不清，导致劳拉在等待 30 分钟后决定取消见面，尽管保罗道歉，劳拉仍拒绝原谅。,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,81,Finn: Hey\r\nZadie: Hi there! What's up?\r\nFi...,Finn 邀请 Zadie 明天下午 2 点前往 Elephant and Castle 购...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,85,Josh: I need to buy an iPad?\r\nJosh: do u thi...,Josh 咨询购买 iPad 的建议，Brian 不推荐苹果品牌，转而推荐三星、小米或索尼，...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,87,Frank: wat are u doing??\r\nAndy: watching Arr...,Frank 提醒 Andy 明天有测验，劝他立即复习，但 Andy 认为测验不重要并计划明天...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,123,Crystal: <file_photo>\r\nIrene: He's so big!\r...,Crystal 分享孩子长大的照片，感叹孩子因长高而衣服不合身。Irene 主动提出带孩子去...,艾琳会带克里斯特尔的儿子去买衣服。
